# Map visualizations

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/ETNA2018")

## Open netCDF files

In [ ]:
import numpy as np
import xarray as xr

fname1 = "data/etna2018-128.ens.nc"
fname2 = output_dir / "posterior-samples.nc"
fname3 = "data/20181224_1800_Meteosat-11_Etna_VPRoutput.nc"

ds1 = xr.open_dataset(fname1)
ds2 = xr.open_dataset(fname2)
ds3 = xr.open_dataset(fname3)

prior_control  = ds1['tephra_col_mass'].isel(ens=0)
prior_mean     = ds1['tephra_col_mass'].mean(dim='ens')
posterior_mean = ds2['mean']
observation    = ds3['mass']

In [ ]:
# Closed loop coordinates for the domain box
lat1 = ds1.lat.minimum
lat2 = ds1.lat.maximum
lon1 = ds1.lon.minimum
lon2 = ds1.lon.maximum
box_lons = [lon1, lon2, lon2, lon1, lon1]
box_lats = [lat1, lat1, lat2, lat2, lat1]

## Panel plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
from modules.plot import get_cbar_size, set_map

In [ ]:
# 1. Initialize subplots
nrows, ncols = 2, 2
fig, axs = plt.subplots(
    nrows=nrows, 
    ncols=ncols,
    subplot_kw={'projection': ccrs.PlateCarree()}, 
    figsize=(14, 10)
)

# 2. Setup discrete levels & consecutive colors mapping
levels = [1E-2, 2E-2, 4E-2, 8E-2, 1E-1, 2E-1, 4E-1, 8E-1, 1, 2, 4]
cmap = plt.get_cmap('viridis', len(levels))
norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=cmap.N, extend='max')

conf = {
    'levels': levels,
    'cmap': cmap,
    'norm': norm,
    'extend': 'max'
}

# Plot contours
_  = axs[0,0].contourf(ds1.lon, ds1.lat, prior_control, **conf)
_  = axs[0,1].contourf(ds1.lon, ds1.lat, prior_mean, **conf)
_  = axs[1,0].contourf(ds1.lon, ds1.lat, posterior_mean, **conf)
cf = axs[1,1].contourf(ds3.longitude, ds3.latitude, observation, **conf)

titles = ['(a) Control run', '(b) Prior mean', '(c) Posterior mean', '(d) Satellite']

for i, ax in enumerate(axs.flat):
    set_map(ax)
    
    ax.set_extent([13.5, 20.5, 34, 39], crs=ccrs.PlateCarree())

    # --- ADD DOMAIN BOUNDING BOX ---
    ax.plot(
        box_lons, 
        box_lats, 
        color='red', 
        linewidth=1.5, 
        linestyle='-', 
        transform=ccrs.PlateCarree(),
        alpha = 0.7,
        zorder=5  # Keep box on top of filled contours
    )

    row, col = np.unravel_index(i, axs.shape)
    active_labels = []
    if col == 0:
        active_labels.append('left')
    if row == nrows - 1:
        active_labels.append('bottom')

    # Gridlines configuration
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=active_labels,
        linewidth=0.5,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )
    
    gl.xlabel_style = {'size': 8}
    gl.ylabel_style = {'rotation': 89, 'size': 8}
    
    # Set titles
    ax.set_title(titles[i])

# Make room on the right for the colorbar
fig.subplots_adjust(right=0.88, wspace=0.04, hspace=0.14)

# [left, bottom, width, height] in figure-fraction coordinates
cbar_ax = fig.add_axes([0.90, 0.25, 0.02, 0.5])
cbar = fig.colorbar(cf, ticks=levels, cax=cbar_ax)
cbar.set_label(r'Ash column mass [$g/m^2$]', fontsize=10)